# FoldMason Conformer Ranking

Ranks Starling conformers using FoldMason JSON output and selects the best ones for downstream binder design.

**Input**: FoldMason JSON file (from `foldmason easy-msa --report-mode 2`)

**Output**: Ranked conformer list + top conformers copied to a `_binder_ready/` folder

## Pipeline position
```
01_starling_idp_ensemble.ipynb
    → _frames_filtered/*.pdb  (Rg-filtered Starling frames)
        → foldmason easy-msa  (run locally or on Colab)
            → foldmason.json
                → THIS NOTEBOOK
                    → _binder_ready/*.pdb  (top conformers for RFDiffusion/BindCraft)
```

## Ranking logic
- **lDDT** (FoldMason): how well the conformer aligns to the MSA consensus — measures structural representativeness
- **Epitope structural content**: fraction of non-D 3Di states at your defined binding epitope residues — measures local structural character at the site you want to design binders against
- **Combined score** = 0.6 × lDDT + 0.4 × epitope structural content


## 0. Install dependencies

In [ ]:
%%capture
!pip install numpy matplotlib pandas scipy

## 1. Configuration

Set your protein name, upload the FoldMason JSON, and define your epitope residue positions.

In [ ]:
# ── EDIT THESE ──────────────────────────────────────────────────────────────

PROTEIN_NAME = "tau_k18"

# Path to FoldMason JSON (upload via the Files panel on the left, or set path)
FOLDMASON_JSON = "foldmason.json"

# Epitope residue positions (1-indexed) — residues you want binders to target
# These are used to score conformers by local structural content at the binding site
# Tau K18 defaults: PHF6* (VQIINK) = 2-7, PHF6 (VQIVYK) = 33-38
EPITOPE_RANGES = [
    (2, 7,  "PHF6* VQIINK"),
    (33, 38, "PHF6 VQIVYK"),
]

# Scoring weights
LDDT_WEIGHT      = 0.6
EPITOPE_WEIGHT   = 0.4

# How many top conformers to keep for binder design
TOP_N = 5

# Directory containing the actual PDB files (from notebook 01)
# Used to copy top conformers to binder_ready/
FRAMES_DIR = "tau_k18_frames_filtered"

# ────────────────────────────────────────────────────────────────────────────

epitope_positions = []
for start, end, label in EPITOPE_RANGES:
    epitope_positions.extend(range(start - 1, end))  # convert to 0-indexed

print(f"Protein      : {PROTEIN_NAME}")
print(f"JSON file    : {FOLDMASON_JSON}")
print(f"Epitope sites: {EPITOPE_RANGES}")
print(f"Top N        : {TOP_N}")

## 2. Load and parse FoldMason JSON

In [ ]:
import json
import numpy as np
import pandas as pd
from collections import Counter

with open(FOLDMASON_JSON) as f:
    data = json.load(f)

entries = data["entries"]
scores  = np.array(data["scores"])
msa_lddt = data["statistics"]["msaLDDT"]

n_conformers = len(entries)
n_residues   = len(entries[0]["aa"])

print(f"Conformers   : {n_conformers}")
print(f"Residues     : {n_residues}")
print(f"Overall MSA lDDT: {msa_lddt:.4f}")
print(f"lDDT  mean={scores.mean():.4f}  std={scores.std():.4f}  min={scores.min():.4f}  max={scores.max():.4f}")
print()

# Verify epitope sequence
seq = entries[0]["aa"]
print("Sequence check at epitope sites:")
for start, end, label in EPITOPE_RANGES:
    print(f"  {label}: {seq[start-1:end]} (positions {start}-{end})")

## 3. Rank conformers

In [ ]:
rows = []
for entry, lddt in zip(entries, scores):
    ss   = entry["ss"]
    name = entry["name"]

    # Structural content at epitope positions
    non_d = sum(1 for p in epitope_positions if p < len(ss) and ss[p] != "D")
    epitope_struct = non_d / len(epitope_positions) if epitope_positions else 0.0

    # 3Di states at each epitope range
    epitope_ss = {}
    for start, end, label in EPITOPE_RANGES:
        epitope_ss[label] = ss[start-1:end]

    combined = LDDT_WEIGHT * lddt + EPITOPE_WEIGHT * epitope_struct

    rows.append({
        "name": name,
        "lddt": round(lddt, 4),
        "epitope_struct": round(epitope_struct, 3),
        "combined": round(combined, 4),
        **{f"3di_{label.split()[0]}": v for label, v in epitope_ss.items()}
    })

df = pd.DataFrame(rows).sort_values("combined", ascending=False).reset_index(drop=True)
df.index += 1  # 1-indexed rank

print(df.head(15).to_string())
print()
print(f"Top pick: {df.iloc[0]['name']}  (lDDT={df.iloc[0]['lddt']}, epitope_struct={df.iloc[0]['epitope_struct']})")

## 4. Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Plot 1: lDDT distribution with top-N highlighted ────────────────────────
ax = axes[0]
top_names = set(df.iloc[:TOP_N]["name"])
colors = ["tomato" if n in top_names else "steelblue" for n in df["name"]]
ax.bar(range(len(df)), df["lddt"].values, color=colors, width=1.0, edgecolor="none")
ax.axhline(msa_lddt, color="black", linestyle="--", linewidth=1, label=f"MSA lDDT={msa_lddt:.3f}")
ax.set_xlabel("Conformer (ranked by combined score)")
ax.set_ylabel("lDDT")
ax.set_title("lDDT per conformer\n(red = top N selected)")
ax.legend(fontsize=8)

# ── Plot 2: lDDT vs epitope structural content scatter ───────────────────────
ax = axes[1]
sc = ax.scatter(
    df["lddt"], df["epitope_struct"],
    c=df["combined"], cmap="RdYlGn",
    s=40, edgecolors="none", alpha=0.85
)
plt.colorbar(sc, ax=ax, label="Combined score")
# Highlight top N
top_df = df.iloc[:TOP_N]
ax.scatter(top_df["lddt"], top_df["epitope_struct"], s=100,
           facecolors="none", edgecolors="black", linewidths=1.5, label=f"Top {TOP_N}")
for _, row in top_df.iterrows():
    ax.annotate(row["name"].replace("pdb",""), (row["lddt"], row["epitope_struct"]),
                fontsize=6.5, ha="left", va="bottom")
ax.set_xlabel("lDDT (ensemble representativeness)")
ax.set_ylabel("Epitope structural content")
ax.set_title("lDDT vs epitope structure\n(ideal: top-right corner)")
ax.legend(fontsize=8)

# ── Plot 3: Per-position 3Di conservation heatmap ───────────────────────────
ax = axes[2]
n_res = len(entries[0]["ss"])
cons = []
for pos in range(n_res):
    col = [e["ss"][pos] for e in entries]
    top_frac = Counter(col).most_common(1)[0][1] / len(col)
    cons.append(top_frac)
cons = np.array(cons)

# Show as a 1D heatmap row, highlight epitope positions
ax.imshow(cons.reshape(1, -1), cmap="RdYlGn", aspect="auto", vmin=0.7, vmax=1.0)
for start, end, label in EPITOPE_RANGES:
    ax.axvspan(start - 1.5, end - 0.5, alpha=0.3, color="blue", label=label)
ax.set_xlabel("Residue position")
ax.set_yticks([])
ax.set_title("Per-position 3Di conservation\n(blue = epitope sites)")
ax.legend(fontsize=7, loc="lower right")

plt.suptitle(f"{PROTEIN_NAME} — FoldMason conformer analysis", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f"{PROTEIN_NAME}_foldmason_ranking.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved foldmason_ranking.png")

## 5. Copy top conformers to binder-ready folder

In [ ]:
import os
import shutil
import glob

binder_dir = f"{PROTEIN_NAME}_binder_ready"
os.makedirs(binder_dir, exist_ok=True)

copied = []
for rank_idx, row in df.iloc[:TOP_N].iterrows():
    frame_name = row["name"]  # e.g. 'frame_0086pdb'
    # Normalize: FoldMason strips the dot so 'frame_0086pdb' -> 'frame_0086.pdb'
    stem = frame_name.replace("pdb", "")
    candidates = glob.glob(os.path.join(FRAMES_DIR, f"{stem}*.pdb")) + \
                 glob.glob(os.path.join(FRAMES_DIR, f"*{stem}*.pdb"))

    if not candidates:
        print(f"  WARNING: PDB not found for {frame_name} in {FRAMES_DIR}/")
        continue

    src = candidates[0]
    dst = os.path.join(binder_dir, f"rank{rank_idx:02d}_{os.path.basename(src)}")
    shutil.copy2(src, dst)
    copied.append(dst)
    print(f"  Rank {rank_idx}: {frame_name}  lDDT={row['lddt']}  epitope={row['epitope_struct']}  → {dst}")

print(f"\n{len(copied)} conformers ready in: {binder_dir}/")
print("Next step: use these PDBs as input to RFDiffusion or BindCraft")

## 6. Save ranking CSV + download

In [ ]:
import zipfile
from google.colab import files

csv_path = f"{PROTEIN_NAME}_conformer_ranking.csv"
df.to_csv(csv_path)
print(f"Ranking saved to {csv_path}")

zip_name = f"{PROTEIN_NAME}_foldmason_results.zip"
with zipfile.ZipFile(zip_name, "w") as zf:
    zf.write(csv_path)
    zf.write(f"{PROTEIN_NAME}_foldmason_ranking.png")
    for f_path in copied:
        zf.write(f_path)

print(f"Downloading {zip_name}")
files.download(zip_name)

## Notes

**3Di structural alphabet** (FoldMason): 20-state structural encoding from FoldSeek. `D` = dominant state in disordered/flexible regions. Non-D states at epitope positions indicate transient local structure — more amenable to binder design.

**lDDT** (FoldMason): measures how consistently a conformer's local distances match the MSA consensus. High lDDT = representative of the ensemble average. Low lDDT = structural outlier.

**Combined score** weights both: a conformer that is representative AND has structural content at your binding site is the best design target.

**Why top 3–5, not just 1?** IDPs are dynamic — designing against multiple conformers improves the chance of finding binders that work across the conformational ensemble. RFDiffusion and BindCraft can each take one PDB at a time; run them on all top conformers and compare ipTM scores downstream.

**Adjusting weights**: if your epitope is very well-characterized and you trust the structural content signal, increase `EPITOPE_WEIGHT` to 0.6. If the IDP is highly disordered everywhere and 3Di content is noisy, weight fully toward lDDT (set `EPITOPE_WEIGHT=0`).
